# Quantum — independent oracles

mpmath 1.3.0 fixture at 200 digits (160 saved). expm for exponentials, eigh for Hermitian matrices. Five exponentials: diagonal, Jordan 2×2/3×3, stiffer Jordan and complex rotation. Three spectra, including eigenvalues of the order of 1e-40.

The test uses relative error on the small eigenvalues: an absolute tolerance of 1e-12 would wrongly accept zero. MatrixExp uses a per-element error normalised by max(1,|reference|); it does not certify relative accuracy for all tiny elements.


In [ ]:
#load "../shared/MathemaSharp.Numerics.csx"
#r "System.Text.Json"
using System;
using System.IO;
using System.Linq;
using System.Numerics;
using System.Text.Json;
using System.Globalization;
using MathemaSharp.Quantum;
var root=new DirectoryInfo(Directory.GetCurrentDirectory());
while(root!=null&&!File.Exists(Path.Combine(root.FullName,"src","MathemaSharp.slnx"))) root=root.Parent;
if(root==null) throw new Exception("Run from repository root");
var oracle=JsonDocument.Parse(File.ReadAllText(Path.Combine(root.FullName,"ops","oracles","numerical-reference.json")));
ComplexMatrix<T> ReadMatrix<T>(JsonElement data) where T:IFloatingPointIeee754<T>,IMinMaxValue<T>
{
    var rows=data.EnumerateArray().ToArray();
    var matrix=new ComplexMatrix<T>(rows.Length,rows[0].GetArrayLength());
    for(var i=0;i<matrix.Rows;i++){var cells=rows[i].EnumerateArray().ToArray();for(var j=0;j<matrix.Cols;j++){var p=cells[j].EnumerateArray().ToArray();matrix[i,j]=new(T.Parse(p[0].GetString(),CultureInfo.InvariantCulture),T.Parse(p[1].GetString(),CultureInfo.InvariantCulture));}}
    return matrix;
}
void Verify<T>(string toleranceText) where T:IFloatingPointIeee754<T>,IMinMaxValue<T>
{
    var tolerance=T.Parse(toleranceText,CultureInfo.InvariantCulture);
    var worst=T.Zero;
    foreach(var row in oracle.RootElement.GetProperty("matrix_exp").EnumerateArray())
    {
        var actual=QuantumAlgorithms.MatrixExp(ReadMatrix<T>(row.GetProperty("matrix")));
        var expected=ReadMatrix<T>(row.GetProperty("expected"));
        for(var i=0;i<actual.Rows;i++) for(var j=0;j<actual.Cols;j++){var error=(actual[i,j]-expected[i,j]).GetMagnitude()/T.Max(T.One,expected[i,j].GetMagnitude());if(error>tolerance)throw new Exception("expm mismatch");worst=T.Max(worst,error);}
    }
    Console.WriteLine(typeof(T).Name+"_expm_max_scaled_error: "+worst.ToString(null,CultureInfo.InvariantCulture));
    foreach(var row in oracle.RootElement.GetProperty("hermitian").EnumerateArray())
    {
        var actual=QuantumAlgorithms.DiagonalizeHermitian2(ReadMatrix<T>(row.GetProperty("matrix")));
        var reference=row.GetProperty("eigenvalues").EnumerateArray().Select(v=>T.Parse(v.GetString(),CultureInfo.InvariantCulture)).ToArray();
        if(T.Abs(actual.LambdaPlus/reference[0]-T.One)>tolerance || T.Abs(actual.LambdaMinus/reference[1]-T.One)>tolerance)throw new Exception("eigenvalue mismatch");
        Console.WriteLine(typeof(T).Name+"_"+row.GetProperty("name").GetString()+"_lambda_minus: "+actual.LambdaMinus.ToString(null,CultureInfo.InvariantCulture));
    }
}
Verify<double>("1e-10");
Verify<Decimal128>("1e-26");
Console.WriteLine("oracle_matrix_exp_and_small_eigenvalues: OK");


Double_expm_max_scaled_error: 1.9761969838327786E-14
Double_diagonal_imbalance_lambda_minus: 1E-40
Double_coupled_imbalance_lambda_minus: 1E-40
Double_complex_hermitian_lambda_minus: 0.20871215252208014
Decimal128_expm_max_scaled_error: 3.38E-32
Decimal128_diagonal_imbalance_lambda_minus: 1E-40
Decimal128_coupled_imbalance_lambda_minus: 1.0E-40
Decimal128_complex_hermitian_lambda_minus: 0.2087121525220799967059764031359957
oracle_matrix_exp_and_small_eigenvalues: OK

## Result and limits
The earlier mean−radius computation cancelled the small eigenvalue even in Decimal128. The diagonal case now keeps the original elements; for the coupled case the determinant/large eigenvalue is used, and FMA to reduce the cancellation of the determinant. Limits remain: underflow, nearly zero determinants and rounded inputs.

Fixture and generator: ops/oracles/numerical-reference.json, ops/generate-numerical-oracles.py. The xUnit tests also include float and Decimal64 for MatrixExp; this notebook shows double and Decimal128. NDSharp is not involved. [Reference](12-api-reference.ipynb), [Roadmap](../../ROADMAP.md).